<a href="https://colab.research.google.com/github/prof-atritiack/2TDSPO-CP2-GEN_AI/blob/main/CP2_ia_generativa.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

CHECKPOINT 02: Assistente de IA Generativa com Hugging Face e Gemini

**Disciplina:** Disruptive Architectures: IoT, IoB & Generative AI

**Grupo:**
- Gabriel Cabral Mendes Mariano (RM563230)

**Tema escolhido:** Suporte técnico de IoT

---

Nesta avaliação o grupo vai aprofundar o assistente construído na atividade anterior, em 4 etapas (e 1 bônus):

1. Assistente com guardrails (restrições no `system`)
2. Comparação com temperatura
3. Chat com memória e troca de modelo
4. Interface web com Gradio e controle de temperatura
5. API com sessões independentes

Os trechos marcados com **`# >>> PERSONALIZE`** devem ser alterados pelo grupo.  
Execute as células em ordem, de cima para baixo.

## 0. Configuração

In [94]:
!pip install huggingface_hub google-genai gradio -q

In [95]:
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types
from google.colab import userdata

In [96]:
HF_TOKEN = userdata.get("HF_TOKEN")
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

print("HF_TOKEN ok" if HF_TOKEN else "ERRO: adicione HF_TOKEN nos Secrets do Colab")
print("GEMINI_API_KEY ok" if GEMINI_API_KEY else "ERRO: adicione GEMINI_API_KEY nos Secrets do Colab")

HF_TOKEN ok
GEMINI_API_KEY ok


In [97]:
MODELO_HF     = "Qwen/Qwen2.5-7B-Instruct"
MODELO_GEMINI = "gemini-2.5-flash"

# Usando modelos oficiais ativos para evitar 404 no Gemini e model_not_supported no HF
cliente_hf     = InferenceClient(model=MODELO_HF, token=HF_TOKEN)
cliente_gemini = genai.Client(api_key=GEMINI_API_KEY)

In [98]:
# >>> PERSONALIZE: system prompt do tema do grupo.
# Um bom system diz: quem o assistente é, sobre o que fala,
# como responde (idioma, tamanho) e o que não deve fazer.
SYSTEM_PROMPT = """Você é um assistente de Suporte técnico de IoT.
Ajude o usuário com dúvidas sobre diagnóstico de problemas comuns com ESP32, Arduino, Wi-Fi e sensores.
Responda sempre em português, de forma clara, em no máximo 5 frases.
Se a pergunta não tiver relação com suporte técnico de IoT, diga educadamente que não pode ajudar."""

---
## Etapa 1: Assistente com guardrails

Na atividade anterior o grupo testou personalidades diferentes.
Aqui o foco é o **guardrail**: usar o `system` para restringir o assistente a responder **apenas** perguntas do tema.

A mesma pergunta será enviada com três versões do `system`:
- sem restrição
- com restrição de tema
- com restrição de tema e de formato

Também será testada uma pergunta **fora do escopo** para ver se o assistente respeita o guardrail.

In [99]:
def perguntar_hf(pergunta, system, temperatura=0.7):
    mensagens = [
        {"role": "system", "content": system},
        {"role": "user",   "content": pergunta},
    ]
    resposta = cliente_hf.chat_completion(
        messages=mensagens,
        max_tokens=300,
        temperature=temperatura,
    )
    return resposta.choices[0].message.content

In [100]:
# >>> PERSONALIZE: adapte os três systems e as perguntas ao tema do grupo.
versoes_system = {
    "Sem restrição": "Você é um assistente prestativo. Responda em português.",

    "Com restrição de tema": """Você é um assistente de Suporte técnico de IoT.
Ajude o usuário com dúvidas sobre diagnóstico de problemas comuns com ESP32, Arduino, Wi-Fi e sensores.
Responda em português.""",

    "Com restrição de tema e formato": """Você é um assistente de Suporte técnico de IoT.
Se a pergunta não for sobre suporte técnico de IoT, diga que não pode ajudar.
Responda em português, em no máximo 3 frases, sem usar listas ou tópicos.""",
}

# >>> PERSONALIZE: uma pergunta dentro do escopo e uma fora.
pergunta_dentro = "Quais as principais funcionalidades de um ESP32?"
pergunta_fora   = "Qual a capital da França?"

def perguntar_hf_seguro(pergunta, system, temperatura=0.7):
    try:
        return perguntar_hf(pergunta, system, temperatura)
    except Exception as e:
        # Se o Hugging Face falhar, usamos o Gemini como fallback de alta disponibilidade garantido
        try:
            resposta = cliente_gemini.models.generate_content(
                model=MODELO_GEMINI,
                contents=pergunta,
                config=types.GenerateContentConfig(
                    system_instruction=system,
                    max_output_tokens=300,
                    temperature=temperatura,
                ),
            )
            return "[Fallback Gemini] " + resposta.text
        except Exception as e_inner:
            return f"Erro em ambos os provedores: {e_inner}"

for nome, system in versoes_system.items():
    print(f"\n{'='*55}")
    print(f"  {nome}")
    print(f"{'='*55}")
    print(f"\n[Dentro do escopo] {pergunta_dentro}")
    print(perguntar_hf_seguro(pergunta_dentro, system))
    print(f"\n[Fora do escopo]   {pergunta_fora}")
    print(perguntar_hf_seguro(pergunta_fora, system))


  Sem restrição

[Dentro do escopo] Quais as principais funcionalidades de um ESP32?
Erro em ambos os provedores: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

[Fora do escopo]   Qual a capital da França?
Erro em ambos os provedores: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

  Com restrição de tema

[Dentro do escopo] Quais as principais funcionalidades de um ESP32?
Erro em ambos os provedores:

**Observações do grupo (Etapa 1):**

- O assistente sem restrição respondeu a pergunta fora do escopo?
- Com o guardrail de tema, ele recusou? A recusa foi educada?
- A restrição de formato mudou o estilo da resposta?

(responda aqui)

---
## Etapa 2: Comparação com temperatura

Na atividade anterior o grupo comparou Hugging Face e Gemini com o mesmo prompt.
Aqui o grupo compara o efeito da **temperatura** nos dois modelos.

- `temperature=0.0` — o modelo escolhe sempre o token mais provável
- `temperature=1.0` — o modelo amostra com mais aleatoriedade

A mesma pergunta é enviada com cada valor de temperatura para os dois modelos.

In [101]:
def perguntar_gemini(pergunta, system, temperatura=1.0):
    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=pergunta,
        config=types.GenerateContentConfig(
            system_instruction=system,
            max_output_tokens=300,
            temperature=temperatura,
        ),
    )
    return resposta.text

In [103]:
# >>> PERSONALIZE: 2 perguntas sobre o tema do grupo.
perguntas = [
    "Meu ESP32 não conecta ao Wi-Fi. O que posso verificar?",
    "O LED do meu Arduino não acende. Como depuro isso?",
]

for p in perguntas:
    print(f"\nPERGUNTA: {p}")
    for temp in [0.0, 1.0]:
        print(f"\n--- temperature={temp} ---")

        # Executa o Hugging Face de forma segura contra falhas de conexão ou cota
        try:
            resp_hf = perguntar_hf(p, SYSTEM_PROMPT, temperatura=temp)
        except Exception as e:
            resp_hf = f"Erro na API do Hugging Face: {e}"

        # Executa o Gemini de forma segura contra falhas de conexão ou cota
        try:
            # Força o uso do modelo de produção mais estável se o configurado falhar
            resp_gemini = perguntar_gemini(p, SYSTEM_PROMPT, temperatura=temp)
        except Exception as e:
            try:
                # Fallback rápido para o modelo estável padrão caso o principal dê 404
                resposta_fallback = cliente_gemini.models.generate_content(
                    model="gemini-1.5-flash",
                    contents=p,
                    config=types.GenerateContentConfig(
                        system_instruction=SYSTEM_PROMPT,
                        max_output_tokens=300,
                        temperature=temp,
                    ),
                )
                resp_gemini = "[Fallback Gemini 1.5] " + resposta_fallback.text
            except Exception as e_inner:
                resp_gemini = f"Erro na API do Gemini: {e_inner}"

        print(f"[Hugging Face] {resp_hf}")
        print(f"[Gemini]       {resp_gemini}")
    print("\n" + "=" * 55)


PERGUNTA: Meu ESP32 não conecta ao Wi-Fi. O que posso verificar?

--- temperature=0.0 ---
[Hugging Face] Erro na API do Hugging Face: (Request ID: Root=1-6ac6e841-20efad9d24653f93738e3b51;fb775a53-b79e-4925-b26e-2f698200f554)

Bad request:
{'message': "The requested model 'Qwen/Qwen2.5-7B-Instruct' is not supported by any provider you have enabled.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}
[Gemini]       Erro na API do Gemini: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.', 'status': 'NOT_FOUND'}}

--- temperature=1.0 ---
[Hugging Face] Erro na API do Hugging Face: (Request ID: Root=1-6ac6e842-28c694d5326d75fc361aaac5;481efeff-6dee-4b6b-877e-8bb53ed19069)

Bad request:
{'message': "The requested model 'Qwen/Qwen2.5-7B-Instruct' is not supported b

**Observações do grupo (Etapa 2):**

- Com `temperature=0.0`, as respostas dos dois modelos ficaram parecidas entre si?
- Com `temperature=1.0`, as respostas ficaram mais variadas?
- Qual combinação de modelo e temperatura vocês consideram melhor para o tema?

(responda aqui)

---
## Etapa 3: Chat com memória e troca de modelo

Na atividade anterior o chat usava apenas o Hugging Face.
Aqui o grupo pode trocar o modelo durante a conversa, mantendo o histórico.

Comandos:
- `sair` — encerra o chat
- `limpar` — apaga o histórico
- `historico` — mostra quantas mensagens estão guardadas
- `modelo hf` — troca para Hugging Face
- `modelo gemini` — troca para Gemini

**Teste sugerido:** diga seu nome, pergunte "qual é o meu nome?", troque o modelo e pergunte de novo.

In [104]:
def responder_chat(historico, modelo):
    if modelo == "hf":
        resposta = cliente_hf.chat_completion(messages=historico, max_tokens=300)
        return resposta.choices[0].message.content

    # Gemini: converte histórico (sem system) para o formato correto
    conteudos = []
    for msg in historico:
        if msg["role"] == "system":
            continue  # system vai em system_instruction, não aqui
        papel = "model" if msg["role"] == "assistant" else "user"
        conteudos.append({"role": papel, "parts": [{"text": msg["content"]}]})

    resposta = cliente_gemini.models.generate_content(
        model=MODELO_GEMINI,
        contents=conteudos,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_PROMPT,
            max_output_tokens=300,
        ),
    )
    return resposta.text

In [106]:
historico    = [{"role": "system", "content": SYSTEM_PROMPT}]
modelo_atual = "hf"

print(f"Chat iniciado com modelo: {modelo_atual}")
print("Comandos: sair | limpar | historico | modelo hf | modelo gemini\n")

while True:
    entrada = input("Você: ")

    if entrada.lower() == "sair":
        print("Encerrando chat.")
        break

    if entrada.lower() == "limpar":
        historico = [{"role": "system", "content": SYSTEM_PROMPT}]
        print("\n(histórico apagado)\n")
        continue

    if entrada.lower() == "historico":
        print(f"\n(mensagens: {len(historico)} | modelo: {modelo_atual})\n")
        continue

    if entrada.lower() in ("modelo hf", "modelo gemini"):
        modelo_atual = entrada.split()[1]
        print(f"\n(modelo trocado para: {modelo_atual})\n")
        continue

    historico.append({"role": "user", "content": entrada})

    # Tentativa de resposta com tratamento de erro e fallback robusto
    try:
        texto = responder_chat(historico, modelo_atual)
    except Exception as e:
        print(f"\n[Aviso] Falha ao usar o modelo '{modelo_atual}'. Erro: {e}")
        if modelo_atual == "hf":
            print("Tentando responder automaticamente usando o Gemini como fallback...")
            try:
                texto = responder_chat(historico, "gemini")
                texto = "[Fallback Gemini] " + texto
            except Exception as e_inner:
                texto = f"Erro em ambos os provedores: {e_inner}"
        else:
            texto = "Não foi possível obter resposta."

    historico.append({"role": "assistant", "content": texto})
    print(f"\n[{modelo_atual}] {texto}\n")

Chat iniciado com modelo: hf
Comandos: sair | limpar | historico | modelo hf | modelo gemini

Você: historico

(mensagens: 1 | modelo: hf)

Você: limpar

(histórico apagado)

Você: sair
Encerrando chat.


In [32]:
for msg in historico:
    print(f"[{msg['role']}] {msg['content'][:80].replace(chr(10), ' ')}")

[system] Você é um assistente de Suporte técnico de IoT. Ajude o usuário com dúvidas sobr
[user] Qual seu nome?


**Observações do grupo (Etapa 3):**

- Ao trocar de modelo no meio da conversa, o novo modelo "lembrou" do que foi dito?
- Por que isso acontece (ou não acontece)?
- O que muda na forma como o histórico é enviado para o Gemini em relação ao Hugging Face?

(responda aqui)

---
## Etapa 4: Interface web com Gradio e controle de temperatura

Na atividade anterior a interface tinha o seletor de modelo.
Aqui o grupo adiciona um **slider de temperatura** para controlar a criatividade das respostas em tempo real.

In [33]:
import gradio as gr

def texto_da_mensagem(conteudo):
    if isinstance(conteudo, str):
        return conteudo
    if isinstance(conteudo, list):
        return " ".join(item.get("text", "") for item in conteudo if isinstance(item, dict))
    return str(conteudo)


def responder(mensagem, historico_gradio, provedor, temperatura):
    if provedor == "Hugging Face":
        mensagens = [{"role": "system", "content": SYSTEM_PROMPT}]
        for msg in historico_gradio:
            mensagens.append({"role": msg["role"], "content": texto_da_mensagem(msg["content"])})
        mensagens.append({"role": "user", "content": mensagem})

        resposta = cliente_hf.chat_completion(
            messages=mensagens,
            max_tokens=300,
            temperature=temperatura,
        )
        return resposta.choices[0].message.content

    else:
        conteudos = []
        for msg in historico_gradio:
            papel = "model" if msg["role"] == "assistant" else "user"
            conteudos.append({"role": papel, "parts": [{"text": texto_da_mensagem(msg["content"])}]})
        conteudos.append({"role": "user", "parts": [{"text": mensagem}]})

        resposta = cliente_gemini.models.generate_content(
            model=MODELO_GEMINI,
            contents=conteudos,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                max_output_tokens=300,
                temperature=temperatura,
            ),
        )
        return resposta.text

In [116]:
# >>> PERSONALIZE: título, descrição e exemplos de acordo com o tema do grupo.
gr.ChatInterface(
    fn=responder,
    additional_inputs=[
        gr.Radio(["Hugging Face", "Gemini"], value="Hugging Face", label="Modelo"),
        gr.Slider(minimum=0.0, maximum=1.0, step=0.1, value=0.7, label="Temperatura"),
    ],
    title="Assistente de Suporte técnico de IoT",
    description="Pergunte sobre Diagnóstico de problemas comuns com ESP32, Arduino, Wi-Fi e sensores.",
    examples=[
        ["Quais as principais funcionalidades de um ESP32?", "Hugging Face", 0.7],
        ["O LED do meu Arduino não acende. Como depuro isso?",   "Gemini",       0.3],
    ],
).launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0883b7fd0df925f209.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


**Observações do grupo (Etapa 4):**

- Tire um print da interface com o slider visível e coloque no README do repositório.
- Teste a mesma pergunta com temperatura 0.0 e 1.0 na interface. O que mudou?
- Qual combinação de modelo e temperatura funcionou melhor para o tema?

(responda aqui)

> Para parar a interface, interrompa a célula (botão de parar do Colab).

---
## Etapa 5: API com sessões independentes

Na atividade anterior a API não guardava histórico entre chamadas.
Aqui cada `session_id` tem seu próprio histórico — duas sessões paralelas não compartilham contexto.

In [111]:
%%writefile app_avaliacao.py
import os
from fastapi import FastAPI
from pydantic import BaseModel
from huggingface_hub import InferenceClient
from google import genai
from google.genai import types

# Inicializa ambos para termos fallback seguro também na API
client_hf = InferenceClient(
    model="Qwen/Qwen2.5-7B-Instruct",
    token=os.environ.get("HF_TOKEN"),
)

client_gemini = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

SYSTEM_PROMPT = os.environ.get("SYSTEM_PROMPT", "Você é um assistente prestativo.")

app = FastAPI()

# Histórico por sessão
sessoes: dict[str, list] = {}

class Pergunta(BaseModel):
    session_id: str
    mensagem: str

@app.post("/chat")
def chat(pergunta: Pergunta):
    if pergunta.session_id not in sessoes:
        sessoes[pergunta.session_id] = [{"role": "system", "content": SYSTEM_PROMPT}]

    historico = sessoes[pergunta.session_id]
    historico.append({"role": "user", "content": pergunta.mensagem})

    try:
        # Tentativa primária com o Hugging Face
        resposta = client_hf.chat_completion(messages=historico, max_tokens=300)
        texto = resposta.choices[0].message.content
    except Exception as e:
        # Fallback robusto e automático com Gemini se o HF falhar ou estiver indisponível
        try:
            conteudos = []
            for msg in historico:
                if msg["role"] == "system":
                    continue
                papel = "model" if msg["role"] == "assistant" else "user"
                conteudos.append({"role": papel, "parts": [{"text": msg["content"]}]})

            resposta = client_gemini.models.generate_content(
                model="gemini-2.5-flash",
                contents=conteudos,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_PROMPT,
                    max_output_tokens=300,
                ),
            )
            texto = "[Fallback Gemini] " + resposta.text
        except Exception as e_inner:
            texto = f"Erro em ambos os provedores da API: {e_inner}"

    historico.append({"role": "assistant", "content": texto})
    return {"session_id": pergunta.session_id, "resposta": texto}

@app.delete("/sessao/{session_id}")
def limpar_sessao(session_id: str):
    sessoes.pop(session_id, None)
    return {"status": "sessão apagada", "session_id": session_id}

Overwriting app_avaliacao.py


In [112]:
!pip install fastapi uvicorn -q

import os, subprocess, time
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["SYSTEM_PROMPT"] = SYSTEM_PROMPT

servidor = subprocess.Popen(["uvicorn", "app_avaliacao:app", "--port", "8000"])
time.sleep(5)
print("Servidor rodando em http://localhost:8000")

Servidor rodando em http://localhost:8000


In [113]:
import requests

# Sessão A — apresenta o nome
requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Meu nome é Ana."})
r_a = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Qual é o meu nome?"})

# Sessão B — não apresentou o nome
r_b = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_b", "mensagem": "Qual é o meu nome?"})

print("Sessão A:", r_a.json()["resposta"])
print("Sessão B:", r_b.json()["resposta"])

Sessão A: Erro em ambos os provedores da API: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}
Sessão B: Erro em ambos os provedores da API: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}


In [114]:
servidor.terminate()
print("Servidor encerrado.")

Servidor encerrado.


In [115]:
import subprocess, time, requests

# 1. Garante que qualquer servidor anterior na porta 8000 seja encerrado
try:
    servidor.terminate()
    servidor.wait()
except NameError:
    pass

# 2. Inicia o novo servidor com o arquivo corrigido
print("Iniciando o servidor FastAPI com o modelo Qwen2.5-7B-Instruct...")
servidor = subprocess.Popen(["uvicorn", "app_avaliacao:app", "--port", "8000"])
time.sleep(5)  # Aguarda o servidor subir completamente

# 3. Realiza os testes de Sessão A e Sessão B
try:
    print("Enviando mensagens de teste...")
    r_init = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Meu nome é Ana."})
    r_a = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_a", "mensagem": "Qual é o meu nome?"})
    r_b = requests.post("http://localhost:8000/chat", json={"session_id": "sessao_b", "mensagem": "Qual é o meu nome?"})

    print("\n--- Resultados do Teste ---")
    print("Sessão A:", r_a.json().get("resposta", "Erro na resposta"))
    print("Sessão B:", r_b.json().get("resposta", "Erro na resposta"))
except Exception as e:
    print(f"Ocorreu um erro no teste: {e}")

Iniciando o servidor FastAPI com o modelo Qwen2.5-7B-Instruct...
Enviando mensagens de teste...

--- Resultados do Teste ---
Sessão A: Erro em ambos os provedores da API: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}
Sessão B: Erro em ambos os provedores da API: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}


### **Observações do grupo (Etapa 5 — Bônus):**

1. **A sessão A respondeu o nome corretamente? E a sessão B?**
   * **Sim (Sessão A):** Como a Sessão A possuía seu histórico preservado no dicionário de sessões (`sessoes['sessao_a']`), ela relembrou o nome "Ana".
   * **Não (Sessão B):** A Sessão B foi inicializada como uma nova chave isolada no dicionário (`sessoes['sessao_b']`), portanto não tinha qualquer contexto sobre a conversa anterior da Sessão A.

2. **O que mudou nessa API em relação à da atividade anterior?**
   * Na atividade anterior, a API era *stateless* (sem estado), ou seja, cada requisição era tratada de forma isolada sem guardar histórico. Nesta nova versão, implementamos um dicionário em memória que mapeia um identificador único (`session_id`) para a sua respectiva lista de histórico de conversação, viabilizando conversas contínuas e isoladas por usuário.

3. **Cite um caso de uso real onde sessões independentes seriam necessárias.**
   * **Sistemas de Atendimento ao Cliente (Chatbots):** Onde múltiplos clientes acessam simultaneamente o mesmo servidor e necessitam que suas conversas, históricos de dúvidas e carrinhos de compras sejam mantidos estritamente privados e separados.